# Lesson 09 — Working Capital and Cash Value

## Goal
Understand how AI workflows can affect **cash**, not only operating expense.

## Learning Objectives

1. Calculate Days Sales Outstanding (DSO), Days Payable Outstanding (DPO), and Cash Conversion Cycle (CCC)
2. Connect invoice processing delays to measurable cash impact (EUR released or trapped)
3. Build a collections prioritisation model that scores AR by risk and value
4. Estimate early payment discount savings from AP automation
5. Construct a working capital AI business case combining AR, AP, and collections interventions

## Prerequisites

- Lesson 06: Cost of Friction Model (friction EUR figures used throughout)
- Lesson 07: Value Stream Metrics (lead time data used for DSO derivation)
- Lesson 08: Graph Bottlenecks to Value (centrality scores referenced in comparison)

## Core Insight

**Friction cost (L06) measures what you spend. Cash trapped in the working capital cycle measures what you cannot use.**
A 7-day DSO improvement on a €4M AR balance releases €78k in cash immediately —
with zero headcount reduction. AI workflows that accelerate invoice intake, prioritise collections,
and optimise payment timing create **balance sheet value**, not just P&L savings.

---

## Part 1 — Working Capital Concepts for AI Engineers

### The Cash Conversion Cycle

```
CCC = DSO + Inventory Days - DPO
```

| Metric | Definition | Formula | AI lever |
|--------|-----------|---------|----------|
| **DSO** | Days to collect cash after invoice | AR / (Revenue / 365) | Faster invoice processing, collections prioritisation |
| **DPO** | Days to pay vendors | AP / (COGS / 365) | Payment term optimiser, early payment discount capture |
| **Inventory Days** | Days of inventory held | Inventory / (COGS / 365) | Demand forecasting, reorder automation |
| **CCC** | Total cash-to-cash cycle | DSO + Inv Days − DPO | Reduce DSO + Inv Days; extend DPO strategically |

### Six Working Capital Questions

1. How many days does it take to collect cash after an invoice is issued?
2. What % of AR is overdue, and which invoices are highest risk?
3. Are we paying vendors faster than required, missing early payment discounts?
4. Which invoice processing delays directly extend DSO?
5. How much cash is currently trapped in the >60-day AR bucket?
6. What is the financial value of a 7-day DSO improvement at our AR scale?

### Connection to L06 / L07 / L08

- **L06**: Invoice friction = €308/quarter (6% of manual cost). This friction causes processing delays that extend DSO.
- **L07**: Invoice lead time = 1.3 days average, but 2% escalation cases take 3–4 days. Extended lead time = extended DSO.
- **L08**: Graph shows Analyst node has highest betweenness centrality. Analyst bottleneck creates AR aging backlog.
- **L09**: Quantifies cash impact of removing those delays.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import date, timedelta

np.random.seed(42)
pd.set_option('display.float_format', '{:,.2f}'.format)
print('Libraries loaded: pandas', pd.__version__, '| numpy', np.__version__)

---

## Part 2 — Working Capital Calculation Utilities

The five functions below are the calculation primitives for this lesson.
They take the same style of inputs as the `cost_of_friction()` function from L06.

| Function | Purpose |
|----------|---------|
| `days_sales_outstanding()` | DSO from AR balance and annual revenue |
| `days_payable_outstanding()` | DPO from AP balance and annual COGS |
| `cash_conversion_cycle()` | CCC = DSO + inventory days - DPO |
| `working_capital_impact()` | EUR cash released or trapped by DSO change |
| `early_payment_discount_value()` | Annual discount savings on 2/10 net 30 terms |

In [ ]:
def days_sales_outstanding(ar_balance, annual_revenue):
    return (ar_balance / annual_revenue) * 365


def days_payable_outstanding(ap_balance, annual_cogs):
    return (ap_balance / annual_cogs) * 365


def cash_conversion_cycle(dso, dpo, inventory_days=0):
    return dso + inventory_days - dpo


def working_capital_impact(ar_balance, annual_revenue, dso_change_days):
    daily_revenue = annual_revenue / 365
    cash_released = daily_revenue * dso_change_days
    return {
        'ar_balance':      ar_balance,
        'annual_revenue':  annual_revenue,
        'daily_revenue':   daily_revenue,
        'dso_change_days': dso_change_days,
        'cash_released':   cash_released,
        'direction':       'released' if dso_change_days > 0 else 'trapped',
    }


def early_payment_discount_value(annual_ap_spend, discount_pct, eligible_pct=0.6):
    eligible_spend = annual_ap_spend * eligible_pct
    annual_saving  = eligible_spend * discount_pct
    annualised_roi = discount_pct / (10 / 365)
    return {
        'annual_ap_spend':     annual_ap_spend,
        'eligible_spend':      eligible_spend,
        'annual_saving':       annual_saving,
        'annualised_roi_pct':  annualised_roi * 100,
    }


print('Working capital functions loaded.')
print('  days_sales_outstanding()')
print('  days_payable_outstanding()')
print('  cash_conversion_cycle()')
print('  working_capital_impact()')
print('  early_payment_discount_value()')

---

## Part 3 — Accounts Receivable: Invoice Processing Delays and DSO

### Context from L06 and L07

The Invoice Approval workflow was analysed in Lessons 06 and 07:

| Metric | Value | Source |
|--------|-------|--------|
| Q1 invoice volume | 1,050 cases | L06/L07 |
| L06 friction cost | €308 (6% of manual cost) | L06 |
| Average lead time | 1.3 days | L07 |
| Escalation rate | 2% (takes 3–4 days) | L07 |
| Rework rate | 5% (additional 24h) | L07 |

**Working capital question:** If invoices are delayed in processing, AR stays
open longer, DSO rises, and cash is trapped. Every extra day in the processing
queue is a day the invoice cannot be sent — or a day it sits unpaid.

### AR Aging Buckets

The AR portfolio below reflects a realistic distribution given L07 lead times.
The >60-day bucket inflates when the Analyst (L08 top bottleneck) is overloaded.

In [ ]:
# AR aging — Q1 Invoice portfolio (1,050 invoices, avg value EUR 4,000)
# Bucket distribution reflects L07 lead time profile

ar_aging = pd.DataFrame({
    'bucket':         ['0-30 days', '31-60 days', '61-90 days', '>90 days'],
    'invoice_count':  [630,          263,           105,           52],
    'avg_value_eur':  [4000,          4000,          4000,          4000],
    'pct_collected':  [0.95,          0.80,          0.50,          0.20],
})

ar_aging['total_ar_eur']    = ar_aging['invoice_count'] * ar_aging['avg_value_eur']
ar_aging['expected_cash']   = ar_aging['total_ar_eur'] * ar_aging['pct_collected']
ar_aging['at_risk_eur']     = ar_aging['total_ar_eur'] * (1 - ar_aging['pct_collected'])
ar_aging['pct_of_total_ar'] = ar_aging['total_ar_eur'] / ar_aging['total_ar_eur'].sum() * 100

total_ar       = ar_aging['total_ar_eur'].sum()
annual_revenue = total_ar * 4   # Q1 x 4 annualised
current_dso    = days_sales_outstanding(total_ar, annual_revenue)

print('AR AGING ANALYSIS — Invoice Portfolio (Q1 2026)')
print('=' * 65)
print(ar_aging[['bucket','invoice_count','total_ar_eur','pct_of_total_ar','at_risk_eur']].to_string(index=False))
print()
print(f'Total AR balance:   EUR {total_ar:>10,.0f}')
print(f'Annual revenue est: EUR {annual_revenue:>10,.0f}  (Q1 x 4)')
print(f'Current DSO:        {current_dso:.1f} days')
print(f'Total at-risk AR:   EUR {ar_aging["at_risk_eur"].sum():>10,.0f}')
print(f'At-risk % of AR:    {ar_aging["at_risk_eur"].sum() / total_ar * 100:.1f}%')

In [ ]:
# AI INTERVENTION: Invoice Intake Assistant
# L07 shows rework (5%) + escalation (2%) add ~1-2 extra days of processing delay
# Invoice intake assistant: validates fields on submission, eliminates rework
# Effect: rework rate 5%->1%, processing delay reduced, >60d AR bucket shrinks

# Before AI: current DSO = current_dso (calculated above)
# After AI: >60d bucket drops from 15% to 5% of invoices
invoice_count_total = ar_aging['invoice_count'].sum()
overdue_before = ar_aging.loc[ar_aging['bucket'].isin(['61-90 days', '>90 days']), 'invoice_count'].sum()
overdue_after  = int(invoice_count_total * 0.05)   # target: 5%

# Recalculate DSO post-intervention
ar_aging_ai = ar_aging.copy()
# Redistribute ~105 invoices from >60d buckets back to 31-60d
reduction = overdue_before - overdue_after
ar_aging_ai.loc[ar_aging_ai['bucket'] == '61-90 days', 'invoice_count'] = overdue_after
ar_aging_ai.loc[ar_aging_ai['bucket'] == '>90 days',   'invoice_count'] = 0
ar_aging_ai.loc[ar_aging_ai['bucket'] == '31-60 days', 'invoice_count'] += reduction
ar_aging_ai['total_ar_eur'] = ar_aging_ai['invoice_count'] * ar_aging_ai['avg_value_eur']

# Weighted avg DSO: assume bucket midpoints 15, 45, 75, 105 days
bucket_midpoints = [15, 45, 75, 105]
dso_before = sum(ar_aging['invoice_count'] * bucket_midpoints) / invoice_count_total
dso_after  = sum(ar_aging_ai['invoice_count'] * bucket_midpoints) / invoice_count_total
dso_improvement = dso_before - dso_after

impact = working_capital_impact(total_ar, annual_revenue, dso_improvement)

print('DSO IMPROVEMENT — Invoice Intake Assistant')
print('=' * 55)
print(f'  Overdue invoices before: {overdue_before} ({overdue_before/invoice_count_total*100:.0f}%)')
print(f'  Overdue invoices after:  {overdue_after}  ({overdue_after/invoice_count_total*100:.0f}%)')
print()
print(f'  Weighted avg DSO before: {dso_before:.1f} days')
print(f'  Weighted avg DSO after:  {dso_after:.1f} days')
print(f'  DSO improvement:         {dso_improvement:.1f} days')
print()
print(f'  AR balance:              EUR {impact["ar_balance"]:,.0f}')
print(f'  Daily revenue:           EUR {impact["daily_revenue"]:,.0f}')
print(f'  Cash released:           EUR {impact["cash_released"]:,.0f}')
print()
print('  L06 FRICTION CONNECTION')
print(f'  Friction cost (Q1):      EUR {308:,.0f}')
print(f'  Cash released (Q1 DSO):  EUR {impact["cash_released"]:,.0f}')
print(f'  Cash:friction ratio:     {impact["cash_released"]/308:.0f}x  <-- cash impact dwarfs friction EUR')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

buckets  = ar_aging['bucket'].tolist()
before_c = ar_aging['invoice_count'].tolist()
after_c  = ar_aging_ai['invoice_count'].tolist()
x = np.arange(len(buckets))
w = 0.35

axes[0].bar(x - w/2, before_c, w, label='Before AI', color='#d73027', alpha=0.85)
axes[0].bar(x + w/2, after_c,  w, label='After AI',  color='#4dac26', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(buckets, rotation=15)
axes[0].set_ylabel('Invoice Count')
axes[0].set_title('AR Aging: Before vs After Invoice Intake Assistant')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# DSO waterfall
stages      = ['Current DSO', 'Rework\neliminated', 'Escalation\nreduced', 'Target DSO']
stage_vals  = [dso_before, -(dso_before - dso_after) * 0.6, -(dso_before - dso_after) * 0.4, dso_after]
bar_colors  = ['#2166ac', '#4dac26', '#4dac26', '#1a9641']
bar_bottoms = [0, dso_before - stage_vals[1], dso_before - stage_vals[1] + stage_vals[2], 0]
axes[1].bar(['Current DSO'], [dso_before], color='#2166ac', alpha=0.85, label=f'Before: {dso_before:.1f}d')
axes[1].bar(['Target DSO'], [dso_after], color='#1a9641', alpha=0.85, label=f'After: {dso_after:.1f}d')
axes[1].axhline(dso_before, color='#d73027', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1].set_ylabel('Days')
axes[1].set_title(f'DSO Improvement: {dso_improvement:.1f} days released')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ar_aging_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'AR Aging: {overdue_before} invoices >60d reduced to {overdue_after} ({int(overdue_after/invoice_count_total*100)}%)')
print(f'Cash impact: EUR {impact["cash_released"]:,.0f} released by {dso_improvement:.1f}-day DSO improvement')

---

## Part 4 — Accounts Payable: DPO and Early Payment Discounts

### The AP Opportunity

Working capital has two sides:

| Side | Direction | AI opportunity |
|------|----------|----------------|
| **AR (receivables)** | Collect cash faster | Invoice intake assistant, collections prioritisation |
| **AP (payables)** | Pay at the right time | Payment term optimiser, early payment discount capture |

Many companies pay vendors faster than required, leaving **early payment discount (EPD) value uncaptured**
or paying before cash is actually needed (reducing DPO unnecessarily).

### 2/10 Net 30 Terms

A vendor offering **2/10 net 30** means:
- Pay within 10 days: receive a 2% discount
- Otherwise: pay full amount within 30 days

The annualised return on capturing that discount:

```
Annualised ROI = discount% / (discount_days / 365) = 2% / (10/365) = 73%
```

**This beats most investment returns.** An AI payment term detector identifies which vendor
invoices have EPD terms and flags them for early payment — capturing the discount systematically.

In [ ]:
# AP vendor payment data — Q1 2026
# Company pays 150 vendors; many have 2/10 net 30 terms but are paid on day 18 avg
ap_data = pd.DataFrame({
    'vendor_tier':    ['Strategic', 'Operational', 'Tail spend', 'Contractors'],
    'vendor_count':   [12,            48,             72,            18],
    'q1_ap_spend':    [480000,        360000,         120000,        90000],
    'avg_pay_day':    [18,             22,              28,            14],
    'standard_terms': [30,             30,              30,            14],
    'epd_eligible':   [True,           True,            False,         False],
    'epd_discount':   [0.02,           0.02,            0.0,           0.0],
    'epd_window_days':[10,             10,               0,             0],
})

annual_ap_spend = ap_data['q1_ap_spend'].sum() * 4
annual_cogs     = annual_ap_spend * 1.4   # COGS typically higher than AP
ap_balance      = ap_data['q1_ap_spend'].sum()
current_dpo     = days_payable_outstanding(ap_balance, annual_cogs)

print('AP PAYMENT PROFILE — Q1 2026')
print('=' * 70)
print(ap_data[['vendor_tier','vendor_count','q1_ap_spend','avg_pay_day','standard_terms','epd_eligible']].to_string(index=False))
print()
print(f'Total Q1 AP spend:  EUR {ap_balance:,.0f}')
print(f'Annualised AP spend: EUR {annual_ap_spend:,.0f}')
print(f'Current DPO:         {current_dpo:.1f} days')
print()
early_payers = ap_data[ap_data['avg_pay_day'] < ap_data['epd_window_days']]
late_payers  = ap_data[(ap_data['avg_pay_day'] > ap_data['epd_window_days']) & ap_data['epd_eligible']]
print(f'Paying before EPD window (missing discount): {len(early_payers)} tiers')
print(f'EPD-eligible tiers paying after EPD window:  {len(late_payers)} tiers')

In [ ]:
# Early payment discount analysis
epd_tiers = ap_data[ap_data['epd_eligible']].copy()

epd_results = []
for _, row in epd_tiers.iterrows():
    result = early_payment_discount_value(
        annual_ap_spend = row['q1_ap_spend'] * 4,
        discount_pct    = row['epd_discount'],
        eligible_pct    = 1.0,
    )
    epd_results.append({
        'vendor_tier':        row['vendor_tier'],
        'annual_spend':       result['annual_ap_spend'],
        'annual_epd_saving':  result['annual_saving'],
        'annualised_roi_pct': result['annualised_roi_pct'],
        'currently_captured': row['avg_pay_day'] <= row['epd_window_days'],
    })

epd_df = pd.DataFrame(epd_results)
captured   = epd_df[epd_df['currently_captured']]['annual_epd_saving'].sum()
uncaptured = epd_df[~epd_df['currently_captured']]['annual_epd_saving'].sum()

print('EARLY PAYMENT DISCOUNT ANALYSIS')
print('=' * 60)
print(epd_df.to_string(index=False))
print()
print(f'Currently captured EPD:   EUR {captured:,.0f}/year')
print(f'Uncaptured EPD (AI opp):  EUR {uncaptured:,.0f}/year')
print(f'Total EPD opportunity:    EUR {captured + uncaptured:,.0f}/year')
print()
print('AI SOLUTION: Payment term anomaly detector')
print('  Reads invoice terms on ingestion, flags EPD-eligible invoices')
print('  Routes to fast-pay queue before EPD window closes')
print(f'  Expected annual saving: EUR {uncaptured * 0.85:,.0f}  (85% capture rate)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: payment day vs EPD window and standard terms
tiers     = ap_data['vendor_tier'].tolist()
avg_pay   = ap_data['avg_pay_day'].tolist()
std_terms = ap_data['standard_terms'].tolist()
epd_win   = ap_data['epd_window_days'].tolist()
x = np.arange(len(tiers))

axes[0].bar(x, avg_pay, color='#2166ac', alpha=0.8, label='Avg pay day')
axes[0].scatter(x, epd_win,   color='#4dac26', zorder=5, s=120, label='EPD window (day 10)', marker='D')
axes[0].scatter(x, std_terms, color='#d73027', zorder=5, s=120, label='Standard terms (day 30)', marker='s')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tiers, rotation=10)
axes[0].set_ylabel('Days after invoice')
axes[0].set_title('AP Payment Timing vs EPD Window')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# Right: EPD captured vs uncaptured
labels  = ['Currently\ncaptured', 'Uncaptured\n(AI opportunity)']
amounts = [captured, uncaptured]
colors  = ['#4dac26', '#d73027']
axes[1].bar(labels, amounts, color=colors, alpha=0.85, width=0.5)
for i, v in enumerate(amounts):
    axes[1].text(i, v + 1000, f'EUR {v:,.0f}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_ylabel('Annual EPD Saving (EUR)')
axes[1].set_title('Early Payment Discount: Captured vs Opportunity')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ap_epd_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'EPD opportunity: EUR {uncaptured:,.0f}/year uncaptured across {len(epd_tiers)} vendor tiers')

---

## Part 5 — Collections Prioritisation: Risk-Score the AR Portfolio

### Why Collections Prioritisation Is a High-Value AI Application

Not all overdue invoices have the same recovery probability.
A collections agent that calls every invoice in date order will:

- Spend time on low-value, easy-to-collect invoices
- Miss high-value invoices at risk of becoming uncollectable
- Lose the window to negotiate before the >90-day bucket

**AI collections prioritisation:**

1. Score each overdue invoice by `overdue_days × invoice_value × customer_risk_factor`
2. Route high-priority invoices to senior collections staff immediately
3. Auto-send reminder letters to low-risk overdue invoices
4. Flag dispute risk: invoices with no PO match or incomplete delivery confirmation

### Expected Impact

| Metric | Before AI | After AI |
|--------|----------|---------|
| Overdue AR rate | 18% of total AR | 8% of total AR |
| Avg days overdue | 52 days | 38 days |
| Write-off rate | 3% of overdue | 1% of overdue |
| Collections cost | 2.1h per invoice | 0.8h per invoice |

In [ ]:
# Collections scoring model — AR portfolio (overdue invoices only)
np.random.seed(42)
n_overdue = 157   # 15% of 1,050 invoices are overdue

overdue_ar = pd.DataFrame({
    'invoice_id':      range(1001, 1001 + n_overdue),
    'customer':        np.random.choice(['Alpha Corp','Beta Ltd','Gamma Inc','Delta SA','Epsilon GmbH'], n_overdue),
    'invoice_value':   np.random.choice([2000, 4000, 6000, 8000, 12000], n_overdue, p=[0.3, 0.35, 0.2, 0.1, 0.05]),
    'days_overdue':    np.random.choice([5, 15, 35, 65, 95], n_overdue, p=[0.3, 0.3, 0.2, 0.12, 0.08]),
    'dispute_flag':    np.random.choice([False, True], n_overdue, p=[0.82, 0.18]),
    'customer_risk':   np.random.choice([1.0, 1.5, 2.0, 3.0], n_overdue, p=[0.5, 0.25, 0.15, 0.10]),
})

# Priority score: higher = collect first
overdue_ar['priority_score'] = (
    overdue_ar['invoice_value'] *
    overdue_ar['days_overdue'] *
    overdue_ar['customer_risk'] / 1000
)

overdue_ar['priority_tier'] = pd.cut(
    overdue_ar['priority_score'],
    bins=[0, 100, 500, 2000, 999999],
    labels=['Low', 'Medium', 'High', 'Critical'],
)

priority_summary = overdue_ar.groupby('priority_tier', observed=True).agg(
    count        = ('invoice_id', 'count'),
    total_eur    = ('invoice_value', 'sum'),
    avg_overdue  = ('days_overdue', 'mean'),
    dispute_pct  = ('dispute_flag', 'mean'),
).reset_index()
priority_summary['dispute_pct'] = priority_summary['dispute_pct'] * 100

print('COLLECTIONS PRIORITY TIERS')
print('=' * 60)
print(priority_summary.to_string(index=False))
print()
print('Top 10 invoices to call TODAY (Critical tier):')
critical = overdue_ar[overdue_ar['priority_tier'] == 'Critical'].sort_values('priority_score', ascending=False).head(10)
print(critical[['invoice_id','customer','invoice_value','days_overdue','dispute_flag','priority_score']].to_string(index=False))

In [ ]:
# Collections AI impact: overdue rate 18% -> 8%, avg overdue days 52 -> 38
# Model: faster collection of Critical/High tiers releases cash

overdue_rate_before = 0.18
overdue_rate_after  = 0.08
avg_overdue_before  = 52
avg_overdue_after   = 38

total_ar_value = ar_aging['total_ar_eur'].sum()

overdue_eur_before = total_ar_value * overdue_rate_before
overdue_eur_after  = total_ar_value * overdue_rate_after
cash_released_coll = overdue_eur_before - overdue_eur_after

# Write-off reduction: 3% -> 1% of overdue invoices
writeoff_before = overdue_eur_before * 0.03
writeoff_after  = overdue_eur_after  * 0.01
writeoff_saving = writeoff_before - writeoff_after

# Collections staff efficiency: 2.1h -> 0.8h per invoice at EUR 45/hr
staff_cost_before = n_overdue * 4 * 2.1 * 45   # quarterly * annualised
staff_cost_after  = n_overdue * 4 * 0.8 * 45
staff_saving      = staff_cost_before - staff_cost_after

print('COLLECTIONS AI AGENT — Impact Summary')
print('=' * 55)
print(f'Overdue AR before:      EUR {overdue_eur_before:>10,.0f}  ({overdue_rate_before*100:.0f}% of AR)')
print(f'Overdue AR after:       EUR {overdue_eur_after:>10,.0f}  ({overdue_rate_after*100:.0f}% of AR)')
print(f'Cash released (AR):     EUR {cash_released_coll:>10,.0f}')
print()
print(f'Write-off before:       EUR {writeoff_before:>10,.0f}')
print(f'Write-off after:        EUR {writeoff_after:>10,.0f}')
print(f'Write-off saving:       EUR {writeoff_saving:>10,.0f}/year')
print()
print(f'Collections staff cost before: EUR {staff_cost_before:>8,.0f}/year')
print(f'Collections staff cost after:  EUR {staff_cost_after:>8,.0f}/year')
print(f'Staff saving:                  EUR {staff_saving:>8,.0f}/year')
print()
print(f'TOTAL ANNUAL VALUE (cash + write-off + staff):')
total_value = writeoff_saving + staff_saving
print(f'  EUR {total_value:,.0f}/year  (excludes one-time cash released of EUR {cash_released_coll:,.0f})')

---

## Part 6 — Cross-Workflow Working Capital Summary

### Connecting L06, L07, L08, and L09

Each lesson adds a new dimension to the same three workflows:

| Lens | Invoice | Support | Reporting |
|------|---------|---------|-----------|
| **L06** Friction EUR | €308 (6%) | €9,603 (27%) | €4,022 (44%) |
| **L07** Lead time | 1.3 days avg | 2.1d / 8.2d escalated | 12–28 days |
| **L08** Top bottleneck | Analyst (0.67 betweenness) | Analyst (0.58) | Executive (0.83) |
| **L09** Cash impact | DSO + EPD + collections | Dispute resolution | Report delays (opportunity cost) |

### Key Insight

**The Invoice workflow has the lowest friction EUR (L06) but the highest cash leverage (L09).**
A small DSO improvement on a large AR balance releases more cash than
eliminating all friction from the Reporting workflow.
This means **cash metrics must be evaluated alongside friction metrics** when prioritising AI investments.

In [ ]:
comparison = pd.DataFrame({
    'workflow':         ['Invoice', 'Support', 'Reporting'],
    'l06_friction_eur': [308,        9603,       4022],
    'l06_friction_pct': [6,          27,          44],
    'l07_lead_time_d':  [1.3,         2.1,        20.0],
    'l08_top_node':     ['Analyst',   'Analyst',  'Executive'],
    'l08_betweenness':  [0.67,         0.58,       0.83],
    'l09_dso_improve_d':[dso_improvement, 0.0,       0.0],
    'l09_cash_eur':     [impact['cash_released'] + cash_released_coll, 0, 0],
    'l09_epd_eur':      [uncaptured * 0.85, 0, 0],
    'ai_intervention':  [
        'Invoice intake + collections agent + EPD detector',
        'Dispute resolution summariser',
        'Automated data collection + draft generation',
    ],
})

comparison['total_annual_value'] = (
    comparison['l06_friction_eur'] * 4 +
    comparison['l09_cash_eur'] +
    comparison['l09_epd_eur']
)

print('CROSS-LESSON WORKING CAPITAL COMPARISON')
print('=' * 80)
cols = ['workflow','l06_friction_eur','l07_lead_time_d','l08_betweenness','l09_dso_improve_d','l09_cash_eur','total_annual_value']
print(comparison[cols].to_string(index=False))
print()
print('KEY PATTERN:')
print('  Invoice: lowest friction EUR but highest cash leverage')
print('  Support: highest friction EUR (L06) but limited direct cash impact')
print('  Reporting: high friction % and long lead time but small AR base')
print()
print('CONCLUSION: Use ALL four lenses to prioritise AI investment.')
print('Friction EUR alone will undervalue Invoice automation by 100x.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: bubble chart — friction EUR (x) vs total value (y), bubble = lead time
workflows = comparison['workflow'].tolist()
x_vals    = comparison['l06_friction_eur'].tolist()
y_vals    = comparison['total_annual_value'].tolist()
sizes     = [comparison.loc[i, 'l07_lead_time_d'] * 40 for i in range(len(comparison))]
colors    = ['#2166ac', '#d73027', '#4dac26']

scatter = axes[0].scatter(x_vals, y_vals, s=sizes, c=colors, alpha=0.75, edgecolors='black', linewidth=1.5)
for i, wf in enumerate(workflows):
    axes[0].annotate(wf, (x_vals[i], y_vals[i]), textcoords='offset points',
                     xytext=(10, 5), fontsize=10, fontweight='bold')
axes[0].set_xlabel('L06 Friction EUR (quarterly)')
axes[0].set_ylabel('Total Annual Value (L06+L09, EUR)')
axes[0].set_title('Friction EUR vs Total Value\n(bubble = L07 lead time days)')
axes[0].grid(alpha=0.3)

# Right: stacked bar — breakdown of value by source
workflows_r   = comparison['workflow'].tolist()
friction_ann  = [v * 4 for v in comparison['l06_friction_eur'].tolist()]
cash_released = comparison['l09_cash_eur'].tolist()
epd_saving    = comparison['l09_epd_eur'].tolist()
x2 = np.arange(len(workflows_r))

b1 = axes[1].bar(x2, friction_ann, label='L06 Friction (annualised)', color='#d73027', alpha=0.8)
b2 = axes[1].bar(x2, cash_released, bottom=friction_ann, label='L09 Cash released', color='#2166ac', alpha=0.8)
b3 = axes[1].bar(x2, epd_saving, bottom=[friction_ann[i]+cash_released[i] for i in range(len(workflows_r))],
                  label='L09 EPD saving', color='#4dac26', alpha=0.8)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(workflows_r)
axes[1].set_ylabel('Annual Value (EUR)')
axes[1].set_title('Value Breakdown by Source: Friction vs Cash vs EPD')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('working_capital_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart shows: Invoice workflow value is dominated by cash levers, not friction EUR')

---

## Part 7 — Exercise 1: Guided — Calculate the CCC for a New Company

A manufacturing company shares the following Q1 figures:

| Metric | Value |
|--------|-------|
| Accounts Receivable balance | EUR 2,800,000 |
| Accounts Payable balance | EUR 980,000 |
| Inventory balance | EUR 1,200,000 |
| Q1 Revenue | EUR 3,500,000 |
| Q1 COGS | EUR 2,100,000 |

**Questions:**

1. Calculate DSO, DPO, inventory days, and CCC
2. The company reduces invoice processing delay from 18h to 6h, improving DSO by 5 days. How much cash is released?
3. A payment term AI identifies EUR 500,000 of AP eligible for 2/10 net 30 terms. What is the annual EPD saving?
4. What is the annualised return on capturing that discount vs paying on day 30?

In [ ]:
# EXERCISE 1 ANSWER KEY
print('EXERCISE 1: Cash Conversion Cycle')
print('=' * 60)
print()

ar_bal  = 2_800_000
ap_bal  = 980_000
inv_bal = 1_200_000
rev_q1  = 3_500_000
cogs_q1 = 2_100_000
annual_rev  = rev_q1 * 4
annual_cogs = cogs_q1 * 4

dso_ex1 = days_sales_outstanding(ar_bal, annual_rev)
dpo_ex1 = days_payable_outstanding(ap_bal, annual_cogs)
inv_days_ex1 = (inv_bal / annual_cogs) * 365
ccc_ex1 = cash_conversion_cycle(dso_ex1, dpo_ex1, inv_days_ex1)

print('Q1 -- CCC Components:')
print(f'  DSO:            {dso_ex1:.1f} days')
print(f'  DPO:            {dpo_ex1:.1f} days')
print(f'  Inventory days: {inv_days_ex1:.1f} days')
print(f'  CCC:            {ccc_ex1:.1f} days  (DSO + Inv - DPO)')
print()

imp_ex1 = working_capital_impact(ar_bal, annual_rev, 5)
print('Q2 -- Cash released by 5-day DSO improvement:')
print(f'  Daily revenue: EUR {imp_ex1["daily_revenue"]:,.0f}')
print(f'  Cash released: EUR {imp_ex1["cash_released"]:,.0f}')
print()

epd_ex1 = early_payment_discount_value(500_000, 0.02, eligible_pct=1.0)
print('Q3 -- Early payment discount saving:')
print(f'  Annual EPD saving: EUR {epd_ex1["annual_saving"]:,.0f}')
print()
print('Q4 -- Annualised return on capturing EPD:')
print(f'  Annualised ROI: {epd_ex1["annualised_roi_pct"]:.0f}%')
print('  (2% return earned in 10 days, annualised = 73%)')
print('  This compares to a typical cost of capital of 8-12%')
print('  => Capturing EPDs is almost always financially optimal')

---

### Exercise 2: Open-Ended — Collections Agent Business Case

A B2B services company has the following AR portfolio:

- Total AR: EUR 6,200,000
- Overdue AR (>30 days): 22% of total
- Average days overdue: 58 days
- Write-off rate: 4% of overdue AR
- Collections team: 3 FTE at EUR 42,000/year each (fully loaded EUR 58,000)
- Time spent per overdue invoice: 2.4 hours
- Invoices in overdue portfolio: 210

The collections AI agent is expected to:
- Reduce overdue rate from 22% to 9%
- Reduce write-off rate from 4% to 1.2%
- Reduce time per invoice from 2.4h to 0.9h

**Questions:**

1. What is the annual write-off saving from the AI agent?
2. How much staff time (FTE equivalent) is freed by the efficiency improvement?
3. What is the total annual financial value (write-off saving + staff saving)?
4. If the AI agent costs EUR 35,000/year, what is the ROI and payback period?

In [ ]:
# EXERCISE 2 ANSWER KEY
print('EXERCISE 2: Collections Agent Business Case')
print('=' * 65)
print()

total_ar_ex2       = 6_200_000
overdue_rate_b_ex2 = 0.22
overdue_rate_a_ex2 = 0.09
writeoff_b_ex2     = 0.04
writeoff_a_ex2     = 0.012
hours_b_ex2        = 2.4
hours_a_ex2        = 0.9
invoices_ex2       = 210
loaded_cost_ex2    = 58_000
agent_cost_ex2     = 35_000

overdue_eur_b_ex2  = total_ar_ex2 * overdue_rate_b_ex2
overdue_eur_a_ex2  = total_ar_ex2 * overdue_rate_a_ex2

wo_before_ex2      = overdue_eur_b_ex2 * writeoff_b_ex2
wo_after_ex2       = overdue_eur_a_ex2 * writeoff_a_ex2
wo_saving_ex2      = wo_before_ex2 - wo_after_ex2

# Staff: collections is quarterly work repeated 4x/year
total_hours_b = invoices_ex2 * 4 * hours_b_ex2
total_hours_a = invoices_ex2 * 4 * hours_a_ex2
hours_freed   = total_hours_b - total_hours_a
fte_freed     = hours_freed / 1800
staff_saving_ex2 = fte_freed * loaded_cost_ex2

total_value_ex2  = wo_saving_ex2 + staff_saving_ex2
net_value_ex2    = total_value_ex2 - agent_cost_ex2
roi_ex2          = net_value_ex2 / agent_cost_ex2
payback_ex2      = agent_cost_ex2 / (total_value_ex2 / 12)

print(f'Q1 -- Write-off saving:')
print(f'  Write-off before: EUR {wo_before_ex2:,.0f}  ({overdue_rate_b_ex2*100:.0f}% overdue x {writeoff_b_ex2*100:.0f}% write-off)')
print(f'  Write-off after:  EUR {wo_after_ex2:,.0f}   ({overdue_rate_a_ex2*100:.0f}% overdue x {writeoff_a_ex2*100:.1f}% write-off)')
print(f'  Annual saving:    EUR {wo_saving_ex2:,.0f}')
print()
print(f'Q2 -- Staff efficiency:')
print(f'  Hours before: {total_hours_b:,.0f} h/year  |  Hours after: {total_hours_a:,.0f} h/year')
print(f'  Hours freed:  {hours_freed:,.0f} h/year  =  {fte_freed:.1f} FTE equivalent')
print(f'  Staff saving: EUR {staff_saving_ex2:,.0f}/year')
print()
print(f'Q3 -- Total annual value: EUR {total_value_ex2:,.0f}/year')
print(f'  Write-off:   EUR {wo_saving_ex2:,.0f}')
print(f'  Staff:       EUR {staff_saving_ex2:,.0f}')
print()
print(f'Q4 -- ROI and payback:')
print(f'  Agent cost:  EUR {agent_cost_ex2:,.0f}/year')
print(f'  Net value:   EUR {net_value_ex2:,.0f}/year')
print(f'  ROI:         {roi_ex2:.1f}x  ({roi_ex2*100:.0f}%)')
print(f'  Payback:     {payback_ex2:.1f} months')

---

### Exercise 3: Capstone — Full Working Capital Business Case

A PE-backed distribution company is reviewing its Order-to-Cash and Procure-to-Pay processes.
You have been given the following data:

**Accounts Receivable:**
- Total AR: EUR 8,400,000 (annualised revenue EUR 42,000,000)
- Overdue AR (>30d): 25% of total AR
- Write-off rate: 3.5% of overdue AR
- Invoice processing time: 22h avg (rework rate 8%, escalation rate 3%)

**Accounts Payable:**
- Annual AP spend: EUR 18,000,000
- EPD-eligible AP: 45% of total at 2/10 net 30 terms
- Currently capturing EPD on: 10% of eligible spend

**AI interventions proposed:**
- Invoice intake assistant: reduces rework 8%->2%, processing 22h->7h, expected DSO -6 days
- Collections prioritisation agent: overdue rate 25%->10%, write-off 3.5%->1%
- Payment term detector: EPD capture rate 10%->70% of eligible spend

**Questions:**

1. Calculate current DSO and cash trapped in overdue AR
2. Calculate the cash released by the invoice intake assistant (DSO -6 days)
3. Calculate annual write-off saving from collections agent
4. Calculate annual EPD saving from payment term detector
5. Build the total business case: combined annual value, implementation cost EUR 120,000, ROI and payback
6. Rank the three interventions by annual value. Which should be implemented first?

In [ ]:
# EXERCISE 3 ANSWER KEY
print('EXERCISE 3: Full Working Capital Business Case')
print('=' * 70)
print()

ar_ex3          = 8_400_000
annual_rev_ex3  = 42_000_000
overdue_b_ex3   = 0.25
overdue_a_ex3   = 0.10
wo_rate_b_ex3   = 0.035
wo_rate_a_ex3   = 0.01
dso_improve_ex3 = 6

annual_ap_ex3   = 18_000_000
epd_eligible_ex3 = 0.45
epd_capture_b_ex3 = 0.10
epd_capture_a_ex3 = 0.70

dso_ex3 = days_sales_outstanding(ar_ex3, annual_rev_ex3)
overdue_eur_b_ex3 = ar_ex3 * overdue_b_ex3
overdue_eur_a_ex3 = ar_ex3 * overdue_a_ex3

print('Q1 -- Current DSO and overdue AR:')
print(f'  Current DSO:     {dso_ex3:.1f} days')
print(f'  Overdue AR:      EUR {overdue_eur_b_ex3:,.0f}  ({overdue_b_ex3*100:.0f}% of AR)')
print()

imp_ex3 = working_capital_impact(ar_ex3, annual_rev_ex3, dso_improve_ex3)
print('Q2 -- Cash released by invoice intake assistant (DSO -6 days):')
print(f'  Daily revenue:   EUR {imp_ex3["daily_revenue"]:,.0f}')
print(f'  Cash released:   EUR {imp_ex3["cash_released"]:,.0f}')
print()

wo_b_ex3 = overdue_eur_b_ex3 * wo_rate_b_ex3
wo_a_ex3 = overdue_eur_a_ex3 * wo_rate_a_ex3
wo_save_ex3 = wo_b_ex3 - wo_a_ex3
print('Q3 -- Write-off saving (collections agent):')
print(f'  Write-off before: EUR {wo_b_ex3:,.0f}')
print(f'  Write-off after:  EUR {wo_a_ex3:,.0f}')
print(f'  Annual saving:    EUR {wo_save_ex3:,.0f}')
print()

eligible_spend_ex3 = annual_ap_ex3 * epd_eligible_ex3
epd_b_ex3 = eligible_spend_ex3 * epd_capture_b_ex3 * 0.02
epd_a_ex3 = eligible_spend_ex3 * epd_capture_a_ex3 * 0.02
epd_save_ex3 = epd_a_ex3 - epd_b_ex3
print('Q4 -- EPD saving (payment term detector):')
print(f'  Eligible AP spend:      EUR {eligible_spend_ex3:,.0f}')
print(f'  EPD saving before (10%): EUR {epd_b_ex3:,.0f}')
print(f'  EPD saving after (70%): EUR {epd_a_ex3:,.0f}')
print(f'  Incremental saving:     EUR {epd_save_ex3:,.0f}')
print()

total_annual_ex3 = wo_save_ex3 + epd_save_ex3
impl_cost_ex3    = 120_000
roi_ex3          = total_annual_ex3 / impl_cost_ex3
payback_ex3      = impl_cost_ex3 / (total_annual_ex3 / 12)
print('Q5 -- Total business case:')
print(f'  Write-off saving:  EUR {wo_save_ex3:,.0f}/year')
print(f'  EPD saving:        EUR {epd_save_ex3:,.0f}/year')
print(f'  Total annual:      EUR {total_annual_ex3:,.0f}/year')
print(f'  Cash released:     EUR {imp_ex3["cash_released"]:,.0f}  (one-time balance sheet impact)')
print(f'  Implementation:    EUR {impl_cost_ex3:,.0f}')
print(f'  ROI:               {roi_ex3:.1f}x  ({roi_ex3*100:.0f}%)')
print(f'  Payback:           {payback_ex3:.1f} months')
print()
print('Q6 -- Priority ranking:')
interventions_ex3 = [
    ('Payment term detector', epd_save_ex3, 'Immediate: software reads terms on ingestion'),
    ('Collections agent',     wo_save_ex3,  'Week 2: score existing AR portfolio'),
    ('Invoice intake asst',   imp_ex3['cash_released'], 'Month 2: validate fields at submission'),
]
interventions_ex3.sort(key=lambda x: x[1], reverse=True)
for rank, (name, value, timing) in enumerate(interventions_ex3, 1):
    print(f'  {rank}. {name}: EUR {value:,.0f}/year  -- {timing}')

---

## Part 8 — Completion Checklist

**You have completed Lesson 09 when you can:**

- [ ] Explain DSO, DPO, inventory days, and CCC in plain language
- [ ] Calculate DSO from an AR balance and revenue figure
- [ ] Quantify the EUR cash released by a DSO improvement
- [ ] Explain why early payment discounts are financially attractive (annualised ROI ~73%)
- [ ] Build an AR aging table and identify which bucket has the highest cash risk
- [ ] Score overdue invoices by priority (value x overdue days x risk factor)
- [ ] Calculate write-off saving from a collections AI intervention
- [ ] Build a combined working capital business case (AR + AP + collections)
- [ ] Explain why friction EUR (L06) and cash EUR (L09) are complementary, not interchangeable
- [ ] Complete Exercises 1, 2, and 3

---

## Key Insights Summary

**Lesson 09: Working Capital and Cash Value**

**Cash impact dwarfs friction EUR at scale** --
A 7-day DSO improvement on a EUR 4M AR balance releases EUR 78k in cash.
The same workflow has only EUR 308 in quarterly friction cost (L06).
Cash and friction measure different things: do not use friction EUR as a proxy for working capital impact.

**The AR aging bucket is the early warning system** --
Invoices in the >60-day bucket are 3x more likely to be written off.
An AI agent that keeps invoices out of that bucket pays for itself through write-off reduction alone.

**Early payment discounts are consistently undervalued** --
A 2/10 net 30 discount has an annualised return of ~73%.
Most companies capture less than 20% of available EPD value due to manual payment routing.

**The four-lens framework (L06 + L07 + L08 + L09):**
- L06: How much does this process cost to run? (friction EUR)
- L07: How long does it take? (lead time, wait time)
- L08: Where are the structural bottlenecks? (centrality)
- L09: What cash is trapped by the delays? (DSO, DPO, write-offs)

**PE implication:** A working capital improvement is a balance sheet event,
not just a P&L event. A EUR 500k cash release improves the debt-to-equity ratio,
increases liquidity, and can reduce net debt at exit — directly affecting valuation.

---

## Next Lesson: Lesson 10 — SaaS and System Dependency Rationalization

In Lesson 10 you will analyse software spend and system dependencies as PE value levers.
You will build a SaaS utilisation model, identify duplicate tools and manual bridges,
and quantify the rationalization opportunity — without over-claiming AI agent replacement.